In [0]:
from pyspark.sql import functions as F

INPUT_PATH = "/Workspace/Users/nabanya04@gmail.com/orders.csv"
DELTA_PATH = "/Volumes/workspace/default/my_vol/supply_chain/cleaned_orders_delta"

1.extract

In [0]:
raw_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)
print(f"Raw rows: {raw_df.count()}")
display(raw_df.limit(5))

Raw rows: 201


order_id,supplier_id,product,quantity,order_date,expected_delivery_date,delivery_date
O0001,S005,Bearings,406.0,2026-01-22,2026-01-30,2026-02-06
O0002,S007,Steel Rods,370.0,2026-07-05,2026-07-19,2026-07-24
O0003,S006,Packaging,283.0,2026-06-07,2026-06-16,2026-06-16
O0004,S002,Fasteners,444.0,2026-04-16,2026-04-27,2026-05-04
O0005,S007,Lubricant,72.0,2026-04-14,2026-04-23,2026-04-25


2.transform

In [0]:
clean_df = (
    raw_df.dropDuplicates(["order_id"])
    .dropna(subset=["order_id", "supplier_id", "quantity", "delivery_date"])
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("expected_delivery_date", F.to_date("expected_delivery_date"))
    .withColumn("delivery_date", F.to_date("delivery_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("delay_days", F.datediff("delivery_date", "expected_delivery_date"))
    .withColumn("is_delayed", (F.col("delay_days") > 0).cast("int"))
)
print(f"Clean rows: {clean_df.count()}")
display(clean_df.limit(5))

Clean rows: 197


order_id,supplier_id,product,quantity,order_date,expected_delivery_date,delivery_date,delay_days,is_delayed
O0015,S006,Lubricant,264,2026-06-22,2026-07-04,2026-07-01,-3,0
O0039,S007,Bearings,223,2026-03-08,2026-03-16,2026-03-13,-3,0
O0041,S006,Steel Rods,356,2026-02-09,2026-02-15,2026-02-22,7,1
O0074,S001,Bearings,267,2026-02-01,2026-02-15,2026-02-17,2,1
O0091,S008,Bearings,256,2026-04-15,2026-04-28,2026-05-05,7,1


3.load

In [0]:
clean_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/my_vol/supply_chain/cleaned_orders_delta")

clean_df.write.format("delta").mode("overwrite").saveAsTable("cleaned_orders")

4.Analysis queries

In [0]:
# top products by quantity ordered
top_products = (
    spark.read.format("delta").load(DELTA_PATH)
    .groupBy("product")
    .agg(F.sum("quantity").alias("total_quantity"))
    .orderBy(F.desc("total_quantity"))
)
display(top_products)

product,total_quantity
Packaging,10728
Bearings,9528
Fasteners,9425
Steel Rods,7323
Circuit Boards,7310
Lubricant,6648
